# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 3: HUẤN LUYỆN, ĐÁNH GIÁ & TRIỂN KHAI MÔ HÌNH (MODEL TRAINING, EVALUATION & DEPLOYMENT)

> **Mục tiêu chính:**
> - Huấn luyện và đánh giá 5-Fold Stratified CV trên 3 thuật toán tự cài đặt (From Scratch): **Multinomial Naive Bayes**, **Logistic Regression (L2)** và **Linear SVM (Hinge Loss)**.
> - Tìm tham số tối ưu bằng `GridSearchCV`.
> - Đánh giá Hold-out Test Set
> - Giải thích 15 từ khóa báo thư spam mạnh nhất và đóng gói mô hình sẵn sàng sử dụng thực tế.

## Khai báo Thư viện & Nạp Dữ liệu Đã Xử lý từ Notebook 2
Nạp các thư viện , dữ liệu `train_test_data.joblib` và cấu trúc `preprocessor.joblib`.

In [ ]:
# Khai báo thư viện & Nạp dữ liệu qua hàm dùng chung load_pipeline_and_splits
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from scripts.feature_engineering import load_pipeline_and_splits
from scripts.model_training import (
    create_scratch_models, build_training_pipelines, run_cross_validation,
    tune_grid_search, get_default_param_grids
)
from scripts.model_evaluation import (
    evaluate_holdout_test, get_top_spam_keywords, export_model_artifacts, select_best_model
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Nạp dây chuyền tiền xử lý và bộ dữ liệu đã chia tách chuẩn hóa
base_preprocessor, X_train, X_test, y_train, y_test = load_pipeline_and_splits()
print(f"Nạp thành công: Train={len(X_train):,} mẫu, Test={len(X_test):,} mẫu")

---
## Mục 6: Huấn luyện mô hình & Tối ưu siêu tham số

### 6.1 Thiết lập 3 mô hình & Kiểm định Chéo Phân tầng (5-Fold Stratified CV)
Đánh giá chéo 5-Fold cho 3 thuật toán trên tập Train:
1. **Naive Bayes:** Thuật toán xác suất đơn giản, tốc độ huấn luyện cực nhanh.
2. **Logistic Regression:** Đo lường trọng số từng đặc trưng, tự động triệt tiêu trọng số từ khóa vô dụng.
3. **Support Vector Machine(Linear SVM):** Thuật toán tìm đường ranh giới phân tách rõ ràng nhất giữa thư thường và thư rác.

In [ ]:
# 3 Pipeline & 5-Fold Stratified CV trên tập Train qua scripts.model_training
models = create_scratch_models(random_state=RANDOM_SEED)

pipelines = build_training_pipelines(base_preprocessor, models=models)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_summary_df = run_cross_validation(pipelines, X_train, y_train, cv=cv, random_state=RANDOM_SEED)

display(cv_summary_df)

**Đúc kết kết quả thi thử (5-Fold CV):**
- Cả 3 mô hình tự cài đặt (From Scratch) đều đạt phong độ vượt trội với F1-Score trên 95% và phương sai cực thấp (±0.19% - 0.24%), chứng tỏ khả năng học bài tổng quát hóa xuất sắc.
- **Linear SVM** (98.12% ± 0.21%) và **Logistic Regression** (97.27% ± 0.24%) thể hiện độ chính xác và tính ổn định vượt trội.
- **Naive Bayes** (95.79% ± 0.19%) huấn luyện siêu tốc (< 0.05s) với Precision rất cao (98.90%), tuy nhiên Recall (92.87%) thấp hơn do giả định các từ khóa độc lập có điều kiện.

### 6.2 Tinh chỉnh tìm siêu tham số tối ưu (GridSearchCV)
Thử nghiệm tự động các tổ hợp tham số (vốn từ vựng của TF-IDF, số từ chọn lọc của Chi-Square và độ nhạy của thuật toán) để đưa từng mô hình lên mức điểm cao nhất.

In [ ]:
# Tinh chỉnh siêu tham số GridSearchCV
models_dir = os.path.join('..', 'models', 'saved_models') if os.path.exists(os.path.join('..', 'models', 'saved_models')) else os.path.join('models', 'saved_models')
os.makedirs(models_dir, exist_ok=True)

param_grids = get_default_param_grids()

tuned_models = tune_grid_search(pipelines, param_grids=param_grids, X_train=X_train, y_train=y_train, cv=cv, models_dir=models_dir)

print(f"Đã lưu 3 mô hình tối ưu vào {models_dir}/")

**Đúc kết sau khi tinh chỉnh siêu tham số:**
- Cả 3 mô hình đều đạt phong độ tối ưu khi lấy **5.000 từ phổ biến** (`max_features = 5000`) và giữ lại **2.000 từ quan trọng nhất** (`select__k = 2000`) kết hợp cùng 7 đặc trưng miền.
- Điểm F1-Score thi thử trên tập Train đều tăng vững chắc:
  - **Linear SVM:** Đạt đỉnh **98.62%** trên tập kiểm tra độc lập.
  - **Logistic Regression:** Đạt **97.60%**.
  - **Naive Bayes:** Đạt **96.03%**.
- Toàn bộ 3 mô hình tối ưu đã được lưu thành các tệp nhị phân `.joblib`  vào thư mục `models/saved_models/`.

---
## Mục 7: Đánh giá Mô hình trên Tập Kiểm tra (Model Evaluation) & Phân tích lỗi

### 7.1 Đánh giá Độc lập trên Tập Kiểm tra (Hold-out Test Set Evaluation)
**Mục tiêu:** Đo lường 4 chỉ số hiệu năng cốt lõi (Accuracy, Precision, Recall, F1-Score) trên 6.532 mẫu kiểm tra độc lập chưa từng xuất hiện trong huấn luyện.

In [ ]:
# Đánh giá trên Hold-out Test Set (6.532 mẫu) qua module scripts.model_evaluation
test_results, cms = evaluate_holdout_test(tuned_models, X_test, y_test)
display(pd.DataFrame(test_results))

**Đúc kết từ Bảng Đánh giá Tập Kiểm tra Độc lập (Hold-out Test Set - 6.532 mẫu):**
- **Support Vector Machine (Linear SVM):** Quán quân toàn diện với **F1-Score 98.67%** (Accuracy 98.73%, Precision 98.71%, Recall 98.62%). Thuật toán Subgradient Descent tối ưu Hinge Loss giúp mô hình tìm ra siêu phẳng phân tách lề tối đại, duy trì độ an toàn tối đa cho người dùng.
- **Logistic Regression (L2):** Đạt sát nút với **F1-Score 97.59%** (Accuracy 97.69%, Precision 97.05%, Recall 98.14%), cơ chế phạt chuẩn hóa L2 kiểm soát hiệu quả các trọng số nhiễu, mang lại khả năng diễn giải xác suất mẫu mực.
- **Naive Bayes:** Đạt **F1-Score 96.03%** (Accuracy 96.34%, Precision 99.31%, Recall 92.96%). Precision đạt mức cực cao (99.31%, chỉ 20 ca bắt nhầm), tuy nhiên Recall đạt 92.96% (219 ca bỏ sót) do giả định từ khóa độc lập bị vi phạm trong thực tế khi các từ ngữ lừa đảo xuất hiện đồng thời.

**Bảng xếp hạng trên đề thi thật (6.532 mẫu Test):**
- **Top 1 - Linear SVM:** F1 đạt **98.67%** (Accuracy 98.73%, Precision 98.71%, Recall 98.62%).
- **Top 2 - Logistic Regression:** F1 đạt **97.59%** (Accuracy 97.69%, Precision 97.05%, Recall 98.14%).
- **Top 3 - Naive Bayes:** Đạt F1 **96.03%** (Accuracy 96.34%, Precision 99.31%, Recall 92.96%). Dù độ chuẩn xác khi báo spam rất cao (Precision 99.31%), nhưng độ bao phủ chỉ đạt 92.96% do vẫn để lọt lưới nhiều thư rác.

### 7.2 Phân tích các ca đoán sai (Confusion Matrix)
Kiểm tra thực tế 2 loại lỗi quan trọng:
- **Bắt nhầm thư quan trọng (FP - False Positive):** Email công việc/thư thường bị đánh nhãn rác.
- **Bỏ sót thư rác (FN - False Negative):** Thư rác lừa đảo lọt qua bộ lọc vào hộp thư chính.

In [ ]:
# Trực quan hóa Ma trận nhầm lẫn cho cả 3 mô hình (Tái sử dụng trực tiếp cms từ evaluate_holdout_test)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = {'naive_bayes': 'Naive Bayes', 'logistic_regression': 'Logistic Regression (L2)', 'linear_svm': 'Support Vector Machine (SVM)'}

for idx, (name, model) in enumerate(tuned_models.items()):
    cm = cms[name]
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Dự đoán Ham', 'Dự đoán Spam'], yticklabels=['Thực tế Ham', 'Thực tế Spam'])
    axes[idx].set_title(titles[name], fontweight='bold')
plt.tight_layout()
plt.show()

**Đúc kết phân tích sai số thực tế (trên 6.532 mẫu Test):**
- **Bắt nhầm thư quan trọng (FP - False Positive):**
  - **Naive Bayes ít bắt nhầm nhất:** Chỉ 20 thư (0.58%), cực kỳ an toàn cho luồng đàm thoại thường.
  - **Linear SVM kiểm soát tốt:** Bắt nhầm 40 thư (1.17%).
  - **Logistic Regression:** Bắt nhầm 93 thư (2.72%).
- **Bỏ sót thư rác (FN - False Negative):**
  - **Linear SVM chặn rác triệt để nhất:** Chỉ để lọt 43 thư rác (1.38%), bảo vệ hộp thư hiệu quả nhất.
  - **Logistic Regression:** Chỉ để lọt 58 thư rác (1.86%).
  - **Naive Bayes bỏ sót nhiều nhất:** Để lọt 219 thư rác (7.04%) do nhược điểm cố hữu của giả định xác suất độc lập.

### 7.3 Giải thích mô hình: Top 15 từ khóa báo hiệu thư rác mạnh nhất
Mở mô hình tốt nhất (Linear SVM) để bóc tách các từ ngữ có trọng số nghi vấn spam lớn nhất.

In [ ]:
# Trích xuất Top 15 từ khóa Spam mạnh nhất từ mô hình quán quân qua scripts.model_evaluation
best_name, best_model = select_best_model(tuned_models, test_results)
top15_df = get_top_spam_keywords(best_model, top_n=15)

plt.figure(figsize=(10, 5))
sns.barplot(data=top15_df, x='Trọng số', y='Từ khóa', palette='Reds_r')
plt.title(f"Top 15 Từ khóa Báo hiệu Thư Rác Mạnh nhất ({best_name})", fontweight='bold')
plt.tight_layout()
plt.show()
display(top15_df)

**Đúc kết về các từ ngữ đặc trưng của thư rác:**
- 15 từ khóa có trọng số lớn nhất phản ánh chính xác 3 chủ đề thư rác phổ biến trong thực tế:
  1. *Lừa đảo tài chính & Thư mạo danh gia tộc/quốc tế:* `prince`, `bismarck`, `mexico`, `american`.
  2. *Biệt dược, sinh lý & Tán tỉnh lừa đảo:* `love`, `increase`, `penis`, `girl`, `sex`, `men`, `man`.
  3. *Hàng giả, đồng hồ nhái & Dịch vụ ngụy tạo:* `replica`, `watches`, `cnn`, `life`.
- Việc các từ khóa ngữ cảnh này dẫn đầu trọng số chứng minh bộ tiền xử lý và kỹ thuật đặc trưng đã bóc tách chính xác bản chất thủ thuật của giới phát tán thư rác.

### 7.4 Đóng gói mô hình và hiện vật đánh giá
- Lưu mô hình tốt nhất thành `best_model.joblib` vào thư mục `models/saved_models/`.
- Xuất tệp thông số kết quả `metadata.json` phục vụ công cụ suy luận tương tác (`notebooks/demo.ipynb`).

In [ ]:
# Lưu best_model và xuất metadata.json phục vụ công cụ suy luận độc lập
models_dir = os.path.join('..', 'models', 'saved_models') if os.path.exists(os.path.join('..', 'models', 'saved_models')) else os.path.join('models', 'saved_models')
os.makedirs(models_dir, exist_ok=True)

best_path, meta_path = export_model_artifacts(tuned_models, test_results, top15_df, models_dir=models_dir)
print(f"Đã lưu best_model.joblib và metadata.json tại {models_dir}/")

**Tổng kết toàn bộ dự án:**
1. **Quy trình chuẩn hóa khép kín:** Hoàn thiện trọn vẹn luồng học máy từ Làm sạch dữ liệu (Phần 1), Kỹ thuật đặc trưng chống rò rỉ (Phần 2), đến Huấn luyện, Tối ưu và Đánh giá chi tiết (Phần 3).
2. **Mô hình xuất sắc:** Thuật toán **Linear SVM** sau khi tối ưu siêu tham số hội tụ đạt hiệu năng vượt trội (F1 98.62%), phân loại chuẩn xác và kiểm soát tốt tỷ lệ bắt nhầm thư quan trọng.
3. **Sẵn sàng triển khai:** Mô hình tối ưu cùng tệp cấu hình `metadata.json` đã được đóng gói hoàn chỉnh, sẵn sàng phục vụ suy luận thời gian thực qua Notebook Demo chuyên dụng (`notebooks/demo.ipynb`).